In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


device = "cuda"
TESTED_MODEL = 'gemma'

model_configs = {
    'dicta': {
        'model_name': "dicta-il/dictalm2.0-instruct",
        'kwargs': {'torch_dtype': torch.bfloat16, 'device_map': device}
    },
    'mistral': {
        'model_name': "mistralai/Mistral-7B-Instruct-v0.2",
        'kwargs': {'torch_dtype': torch.bfloat16, 'device_map': device}
    },
    'gemma': {
        'model_name': "google/gemma-2-9b-it",
        'kwargs': {'torch_dtype': torch.bfloat16, 'device_map': device, 'token': 'YOUR_TOKEN'}
    }
}

if TESTED_MODEL in model_configs:
    config = model_configs[TESTED_MODEL]
    model = AutoModelForCausalLM.from_pretrained(config['model_name'], **config['kwargs'])
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], **config['kwargs'])
else:
    raise Exception("Please choose a supported model.")

# Const (prompts)

In [ ]:
# ## Choosing random prompts

# import random 
# import json
# with open('Lchaim_project/aws_heb_train.json', 'r') as f:
#     ds = json.loads(f.read())

    

# random.seed(42)

# short_premises = []
# for d in ds:
#     if d['premise'] and 150 < len(d['premise']) < 210:
#         short_premises.append(d)



# random.sample(short_premises,3)


In [ ]:
####################CONTROL####################

ENGLISH_ZERO_SHOT_PROMPT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        responses examples:
        response:
        E
        response:
        C
        response:
        N
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:"""
ENGLISH_ONE_SHOT_PROMPT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        100 Years of the Western Workplace Conditions in the working environment of Western countries changed significantly over the 20th century. Though not without some associated problems, these changes may be viewed generally as positive: child labour all but ceased, wages rose, the number of working hours in a week decreased, pension policies became standard, fringe benefits multiplied and concerns over health and safety issues were enforced. The collection of data relating to work conditions also became a far more exact science. In particular, there were important developments in methodology and data gathering. Additionally, there was a major expansion of the data collection effort more people became involved in learning about the workplace; and, for the first time, results started to be published. This being the case, at the end of the century, not only were most workers better off than their early 20th century predecessors had been, but they were also in a position to understand how and why this was the case. By carefully analyzing the statistical data made available, specific changes in the workplace not least regarding the concept of what work should involve became clearly discernible. The most obvious changes to the workplace involved the size and composition of the countries workforces. Registering only 24 million in 1900 (and including labourers of age ten and up) and 139 million (aged 16 and older), the size of Americas workforce, for instance, increased by almost six-fold in line with its overall population growth. At the same time, the composition of the workforce shifted from industries dominated by primary production occupations, such as farmers and foresters, to those dominated by professional, technical and, in particular, service workers. At the beginning of the 20th century, 38% of all American workers were employed on farms, by the end of the same century, that figure had fallen to less than 3 %. In Europe, much the same process occurred. In the 1930s, in every European country, bar Britain and Belgium, more than 20 per cent of the population worked in agriculture. By the 1980s, however, the farming populations of all developed countries, excluding Eastern Europe, had dropped to ten per cent and often even lower. At the same time, capital intensive farming using highly mechanized techniques dramatically reduced the numbers needed to farm there. And therein lay the problem. While the workplace became a safer and more productive environment, a world away from the harsh working conditions of our forefathers, the switch from an agricultural to a modern working environment also created massive unemployment in many countries. Fundamental to this problem was the widespread move from the countryside to the city. Having lost their livelihoods, the worlds peasant populations amassed in ever larger numbers in already crowded communities, where rates of job growth failed to keep up with internal migration. As a result, thousands were left squatting in shanty towns on the periphery of cities, waiting for jobs that might never arrive. While this was (and is) particularly true of Third World countries, the same phenomenon could also be witnessed in several American, French, English and German cities in the late 20th century. From a different and more positive perspective, in the 20th century, women became visible and active members of all sectors of the Western workplace. In 1900, only 19% of European women of working age participated in the labour force; by 1999, this figure had risen to 60%. In 1900, only 1% of the countrys lawyers and 6% of its physicians were female; by contrast, the figures were 29% and 24% in 1999. A recent survey of French teenagers, both male and female, revealed that over 50% of those polled thought that, in any job (bar those involving military service), women make better employees, as they are less likely to become riled under stress and less overtly competitive than men. The last and perhaps most significant change to the 20th-century workplace involved the introduction of technology. The list of technological improvements in the workplace is endless: communication and measuring devices, computers of all shapes and sizes, x-ray, lasers, neon lights, stainless steel, and so on and on. Such improvements led to a more productive, safer work environment. Moreover, the fact that medicine improved so dramatically led to an increase in the average lifespan among Western populations. In turn, workers of very different ages were able to work shoulder to shoulder, and continue in their jobs far longer. By the end of 20th century, the Western workplace had undergone remarkable changes. In general, both men and women worked fewer hours per day for more years under better conditions. Yet, the power of agriculture had waned as farmers and foresters moved to cities to earn greater salaries as annalists and accountants. For those who could not make this transition, however, life at the dawn of the new century seemed less appealing.
        Sentence:
        Improvements in medicine led to workers earning more over a longer period.
        Response:
        n
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
ENGLISH_TWO_SHOT_PROMPT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        100 Years of the Western Workplace Conditions in the working environment of Western countries changed significantly over the 20th century. Though not without some associated problems, these changes may be viewed generally as positive: child labour all but ceased, wages rose, the number of working hours in a week decreased, pension policies became standard, fringe benefits multiplied and concerns over health and safety issues were enforced. The collection of data relating to work conditions also became a far more exact science. In particular, there were important developments in methodology and data gathering. Additionally, there was a major expansion of the data collection effort more people became involved in learning about the workplace; and, for the first time, results started to be published. This being the case, at the end of the century, not only were most workers better off than their early 20th century predecessors had been, but they were also in a position to understand how and why this was the case. By carefully analyzing the statistical data made available, specific changes in the workplace not least regarding the concept of what work should involve became clearly discernible. The most obvious changes to the workplace involved the size and composition of the countries workforces. Registering only 24 million in 1900 (and including labourers of age ten and up) and 139 million (aged 16 and older), the size of Americas workforce, for instance, increased by almost six-fold in line with its overall population growth. At the same time, the composition of the workforce shifted from industries dominated by primary production occupations, such as farmers and foresters, to those dominated by professional, technical and, in particular, service workers. At the beginning of the 20th century, 38% of all American workers were employed on farms, by the end of the same century, that figure had fallen to less than 3 %. In Europe, much the same process occurred. In the 1930s, in every European country, bar Britain and Belgium, more than 20 per cent of the population worked in agriculture. By the 1980s, however, the farming populations of all developed countries, excluding Eastern Europe, had dropped to ten per cent and often even lower. At the same time, capital intensive farming using highly mechanized techniques dramatically reduced the numbers needed to farm there. And therein lay the problem. While the workplace became a safer and more productive environment, a world away from the harsh working conditions of our forefathers, the switch from an agricultural to a modern working environment also created massive unemployment in many countries. Fundamental to this problem was the widespread move from the countryside to the city. Having lost their livelihoods, the worlds peasant populations amassed in ever larger numbers in already crowded communities, where rates of job growth failed to keep up with internal migration. As a result, thousands were left squatting in shanty towns on the periphery of cities, waiting for jobs that might never arrive. While this was (and is) particularly true of Third World countries, the same phenomenon could also be witnessed in several American, French, English and German cities in the late 20th century. From a different and more positive perspective, in the 20th century, women became visible and active members of all sectors of the Western workplace. In 1900, only 19% of European women of working age participated in the labour force; by 1999, this figure had risen to 60%. In 1900, only 1% of the countrys lawyers and 6% of its physicians were female; by contrast, the figures were 29% and 24% in 1999. A recent survey of French teenagers, both male and female, revealed that over 50% of those polled thought that, in any job (bar those involving military service), women make better employees, as they are less likely to become riled under stress and less overtly competitive than men. The last and perhaps most significant change to the 20th-century workplace involved the introduction of technology. The list of technological improvements in the workplace is endless: communication and measuring devices, computers of all shapes and sizes, x-ray, lasers, neon lights, stainless steel, and so on and on. Such improvements led to a more productive, safer work environment. Moreover, the fact that medicine improved so dramatically led to an increase in the average lifespan among Western populations. In turn, workers of very different ages were able to work shoulder to shoulder, and continue in their jobs far longer. By the end of 20th century, the Western workplace had undergone remarkable changes. In general, both men and women worked fewer hours per day for more years under better conditions. Yet, the power of agriculture had waned as farmers and foresters moved to cities to earn greater salaries as annalists and accountants. For those who could not make this transition, however, life at the dawn of the new century seemed less appealing.
        Sentence:
        Improvements in medicine led to workers earning more over a longer period.
        Response:
        n
        Paragraph:
        A Disaster of Titanic Proportions At 11:39 p. m. on the evening of Sunday, 14 April 1912, lookouts Frederick Fleet and Reginald Lee on the forward mast of the Titanic sighted an eerie, black mass coming into view directly in front of the ship. Fleet picked up the phone to the helm, waited for Sixth Officer Moody to answer, and yelled Iceberg, right ahead! The greatest disaster in maritime history was about to be set in motion. Thirty-seven seconds later, despite the efforts of officers in the bridge and engine room to steer around the iceberg, the Titanic struck a piece of submerged ice, bursting rivets in the ships hull and flooding the first five watertight compartments. The ships designer, Thomas Andrews, carried out a visual inspection of the ships damage and informed Captain Smith at midnight that the ship would sink in less than two hours. By 1 2:30 a. m. , the lifeboats were being filled with women and children, after Smith had given the command for them to be uncovered and swung out 15 minutes earlier. The first lifeboat was successfully lowered 15 minutes later, with only 28 of its 65 seats occupied. By 1:15 a. m. , the waterline was beginning to reach the Titanics name on the ships bow, and over the next hour, every lifeboat would be released as officers struggled to maintain order amongst the growing panic on board. The dosing moments of the Titanics sinking began shortly after 2 a. m. , as the last lifeboat was lowered and the ships propellers lifted out of the water, leaving the 1,500 passengers still on board to surge towards the stern. At 2:17 a. m. , Harold Bride and Jack Philips tapped out their last wireless message after being relieved of duty as the ships wireless operators, and the ships band stopped playing. Less than a minute later, occupants of the lifeboats witnessed the ships lights flash once, then go black, and a huge roar signalled the Titanics contents plunging towards the bow, causing the front half of the ship to break off and go under. The Titanics stem bobbed up momentarily, and at 2:20 a. m. , the ship finally disappeared beneath the frigid waters. What or who was responsible for the scale of this catastrophe? Explanations abound, some that focus on very small details. Due to a last-minute change in the ships officer line-up, iceberg lookouts Frederick Fleet and Reginald Lee were making do without a pair of binoculars that an officer transferred off the ship in Southampton had left in a cupboard onboard, unbeknownst to any of the ships crew. Fleet, who survived the sinking, insisted at a subsequent inquiry that he could have identified the iceberg in time to avert disaster if he had been in possession of the binoculars. Less than an hour before the Titanic struck the iceberg, wireless operator Cyril Evans on the California, located just 20 miles to the north, tried to contact operator Jack Philips on the Titanic to warn him of pack ice in the area. Shut up, shut up, youre jamming my signal, Philips replied. Im busy. The Titanics wireless system had broken down for several hours earlier that day, and Philips was clearing a backlog of personal messages that passengers had requested to be sent to family and friends in the USA. Nevertheless, Captain Smith had maintained the ships speed of 22 knots despite multiple earlier warnings of ice ahead. It has been suggested that Smith was under pressure to make headlines by arriving early in New York, but maritime historians such as Richard Howell have countered this perception, noting that Smith was simply following common procedure at the time, and not behaving recklessly. One of the strongest explanations for the severe loss of life has been the fact that the Titanic did not carry enough lifeboats for everyone on board. Maritime regulations at the time tied lifeboat capacity to the ship size, not to the number of passengers on board. This meant that the Titanic, with room for 1,178 of its 2,222 passengers, actually surpassed the Board of Trades requirement that it carry lifeboats for 1,060 of its passengers. Nevertheless, with lifeboats being lowered less than half full in many cases, and only 71 2 passengers surviving despite a two-and-a-half-hour window of opportunity, more lifeboats would not have guaranteed more survivors in the absence of better training and preparation. Many passengers were confused about where to go after the order to launch lifeboats was given; a lifeboat drill scheduled for earlier on the same day that the Titanic struck the iceberg was cancelled by Captain Smith in order to allow passengers to attend church.
        Sentence:
        Howell believed the captains failure to reduce speed was an irresponsible action.
        Response:
        c
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
ENGLISH_THREE_SHOT_PROMPT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        100 Years of the Western Workplace Conditions in the working environment of Western countries changed significantly over the 20th century. Though not without some associated problems, these changes may be viewed generally as positive: child labour all but ceased, wages rose, the number of working hours in a week decreased, pension policies became standard, fringe benefits multiplied and concerns over health and safety issues were enforced. The collection of data relating to work conditions also became a far more exact science. In particular, there were important developments in methodology and data gathering. Additionally, there was a major expansion of the data collection effort more people became involved in learning about the workplace; and, for the first time, results started to be published. This being the case, at the end of the century, not only were most workers better off than their early 20th century predecessors had been, but they were also in a position to understand how and why this was the case. By carefully analyzing the statistical data made available, specific changes in the workplace not least regarding the concept of what work should involve became clearly discernible. The most obvious changes to the workplace involved the size and composition of the countries workforces. Registering only 24 million in 1900 (and including labourers of age ten and up) and 139 million (aged 16 and older), the size of Americas workforce, for instance, increased by almost six-fold in line with its overall population growth. At the same time, the composition of the workforce shifted from industries dominated by primary production occupations, such as farmers and foresters, to those dominated by professional, technical and, in particular, service workers. At the beginning of the 20th century, 38% of all American workers were employed on farms, by the end of the same century, that figure had fallen to less than 3 %. In Europe, much the same process occurred. In the 1930s, in every European country, bar Britain and Belgium, more than 20 per cent of the population worked in agriculture. By the 1980s, however, the farming populations of all developed countries, excluding Eastern Europe, had dropped to ten per cent and often even lower. At the same time, capital intensive farming using highly mechanized techniques dramatically reduced the numbers needed to farm there. And therein lay the problem. While the workplace became a safer and more productive environment, a world away from the harsh working conditions of our forefathers, the switch from an agricultural to a modern working environment also created massive unemployment in many countries. Fundamental to this problem was the widespread move from the countryside to the city. Having lost their livelihoods, the worlds peasant populations amassed in ever larger numbers in already crowded communities, where rates of job growth failed to keep up with internal migration. As a result, thousands were left squatting in shanty towns on the periphery of cities, waiting for jobs that might never arrive. While this was (and is) particularly true of Third World countries, the same phenomenon could also be witnessed in several American, French, English and German cities in the late 20th century. From a different and more positive perspective, in the 20th century, women became visible and active members of all sectors of the Western workplace. In 1900, only 19% of European women of working age participated in the labour force; by 1999, this figure had risen to 60%. In 1900, only 1% of the countrys lawyers and 6% of its physicians were female; by contrast, the figures were 29% and 24% in 1999. A recent survey of French teenagers, both male and female, revealed that over 50% of those polled thought that, in any job (bar those involving military service), women make better employees, as they are less likely to become riled under stress and less overtly competitive than men. The last and perhaps most significant change to the 20th-century workplace involved the introduction of technology. The list of technological improvements in the workplace is endless: communication and measuring devices, computers of all shapes and sizes, x-ray, lasers, neon lights, stainless steel, and so on and on. Such improvements led to a more productive, safer work environment. Moreover, the fact that medicine improved so dramatically led to an increase in the average lifespan among Western populations. In turn, workers of very different ages were able to work shoulder to shoulder, and continue in their jobs far longer. By the end of 20th century, the Western workplace had undergone remarkable changes. In general, both men and women worked fewer hours per day for more years under better conditions. Yet, the power of agriculture had waned as farmers and foresters moved to cities to earn greater salaries as annalists and accountants. For those who could not make this transition, however, life at the dawn of the new century seemed less appealing.
        Sentence:
        Improvements in medicine led to workers earning more over a longer period.
        Response:
        n
        Paragraph:
        A Disaster of Titanic Proportions At 11:39 p. m. on the evening of Sunday, 14 April 1912, lookouts Frederick Fleet and Reginald Lee on the forward mast of the Titanic sighted an eerie, black mass coming into view directly in front of the ship. Fleet picked up the phone to the helm, waited for Sixth Officer Moody to answer, and yelled Iceberg, right ahead! The greatest disaster in maritime history was about to be set in motion. Thirty-seven seconds later, despite the efforts of officers in the bridge and engine room to steer around the iceberg, the Titanic struck a piece of submerged ice, bursting rivets in the ships hull and flooding the first five watertight compartments. The ships designer, Thomas Andrews, carried out a visual inspection of the ships damage and informed Captain Smith at midnight that the ship would sink in less than two hours. By 1 2:30 a. m. , the lifeboats were being filled with women and children, after Smith had given the command for them to be uncovered and swung out 15 minutes earlier. The first lifeboat was successfully lowered 15 minutes later, with only 28 of its 65 seats occupied. By 1:15 a. m. , the waterline was beginning to reach the Titanics name on the ships bow, and over the next hour, every lifeboat would be released as officers struggled to maintain order amongst the growing panic on board. The dosing moments of the Titanics sinking began shortly after 2 a. m. , as the last lifeboat was lowered and the ships propellers lifted out of the water, leaving the 1,500 passengers still on board to surge towards the stern. At 2:17 a. m. , Harold Bride and Jack Philips tapped out their last wireless message after being relieved of duty as the ships wireless operators, and the ships band stopped playing. Less than a minute later, occupants of the lifeboats witnessed the ships lights flash once, then go black, and a huge roar signalled the Titanics contents plunging towards the bow, causing the front half of the ship to break off and go under. The Titanics stem bobbed up momentarily, and at 2:20 a. m. , the ship finally disappeared beneath the frigid waters. What or who was responsible for the scale of this catastrophe? Explanations abound, some that focus on very small details. Due to a last-minute change in the ships officer line-up, iceberg lookouts Frederick Fleet and Reginald Lee were making do without a pair of binoculars that an officer transferred off the ship in Southampton had left in a cupboard onboard, unbeknownst to any of the ships crew. Fleet, who survived the sinking, insisted at a subsequent inquiry that he could have identified the iceberg in time to avert disaster if he had been in possession of the binoculars. Less than an hour before the Titanic struck the iceberg, wireless operator Cyril Evans on the California, located just 20 miles to the north, tried to contact operator Jack Philips on the Titanic to warn him of pack ice in the area. Shut up, shut up, youre jamming my signal, Philips replied. Im busy. The Titanics wireless system had broken down for several hours earlier that day, and Philips was clearing a backlog of personal messages that passengers had requested to be sent to family and friends in the USA. Nevertheless, Captain Smith had maintained the ships speed of 22 knots despite multiple earlier warnings of ice ahead. It has been suggested that Smith was under pressure to make headlines by arriving early in New York, but maritime historians such as Richard Howell have countered this perception, noting that Smith was simply following common procedure at the time, and not behaving recklessly. One of the strongest explanations for the severe loss of life has been the fact that the Titanic did not carry enough lifeboats for everyone on board. Maritime regulations at the time tied lifeboat capacity to the ship size, not to the number of passengers on board. This meant that the Titanic, with room for 1,178 of its 2,222 passengers, actually surpassed the Board of Trades requirement that it carry lifeboats for 1,060 of its passengers. Nevertheless, with lifeboats being lowered less than half full in many cases, and only 71 2 passengers surviving despite a two-and-a-half-hour window of opportunity, more lifeboats would not have guaranteed more survivors in the absence of better training and preparation. Many passengers were confused about where to go after the order to launch lifeboats was given; a lifeboat drill scheduled for earlier on the same day that the Titanic struck the iceberg was cancelled by Captain Smith in order to allow passengers to attend church.
        Sentence:
        Howell believed the captains failure to reduce speed was an irresponsible action.
        Response:
        c
        Paragraph:
        A European spacecraft took off today to spearhead the search for another "Earth" among the stars. The Corot space telescope blasted off aboard a Russian Soyuz rocket from the Baikonur cosmodrome in Kazakhstan shortly after 2.20pm. Corot, short for convection rotation and planetary transits, is the first instrument capable of finding small rocky planets beyond the solar system. Any such planet situated in the right orbit stands a good chance of having liquid water on its surface, and quite possibly life, although a leading scientist involved in the project said it was unlikely to find "any little green men". Developed by the French space agency, CNES, and partnered by the European Space Agency (ESA), Austria, Belgium, Germany, Brazil and Spain, Corot will monitor around 120,000 stars with its 27cm telescope from a polar orbit 514 miles above the Earth. Over two and a half years, it will focus on five to six different areas of the sky, measuring the brightness of about 10,000 stars every 512 seconds. "At the present moment we are hoping to find out more about the nature of planets around stars which are potential habitats. We are looking at habitable planets, not inhabited planets. We are not going to find any little green men, " Professor Ian Roxburgh, an ESA scientist who has been involved with Corot since its inception, told the BBC Radio 4 Today programme. Prof Roxburgh said it was hoped Corot would find "rocky planets that could develop an atmosphere and, if they are the right distance from their parent star, they could have water". To search for planets, the telescope will look for the dimming of starlight caused when an object passes in front of a star, known as a "transit". Although it will take more sophisticated space telescopes planned in the next 10 years to confirm the presence of an Earth-like planet with oxygen and liquid water, Corot will let scientists know where to point their lenses. Measurements of minute changes in brightness will enable scientists to detect giant Jupiter-like gas planets as well as small rocky ones. It is the rocky planets - that could be no bigger than about twice the size of the Earth - which will cause the most excitement. Scientists expect to find between 10 and 40 of these smaller planets. Corot will also probe into stellar interiors by studying the acoustic waves that ripple across the surface of stars, a technique called "asteroseismology". The nature of the ripples allows astronomers to calculate a stars precise mass, age and chemical composition. "A planet passing in front of a star can be detected by the fall in light from that star. Small oscillations of the star also produce changes in the light emitted, which reveal what the star is made of and how they are structured internally. This data will provide a major boost to our understanding of how stars form and evolve, " Prof Roxburgh said. Since the discovery in 1995 of the first "exoplanet" - a planet orbiting a star other than the Sun - more than 200 others have been found by ground-based observatories. Until now the usual method of finding exoplanets has been to detect the "wobble" their gravity imparts on parent stars. But only giant gaseous planets bigger than Jupiter can be found this way, and they are unlikely to harbour life. In the 2010s, ESA plans to launch Darwin, a fleet of four or five interlinked space telescopes that will not only spot small rocky planets, but analyse their atmospheres for signs of biological activity. At around the same time, the US space agency, Nasa, will launch Terrestrial Planet Finder, another space telescope designed to locate Earth-like planets.
        Sentence:
        Scientists are trying to find out about the planets that can be inhabited.
        Response:
        e
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """

###### SHORT ######

ENGLISH_ONE_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        A person on a horse jumps over a broken down airplane.
        Sentence:
        A person is training his horse for a competition.
        Response:
        n
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
ENGLISH_TWO_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        A person on a horse jumps over a broken down airplane.
        Sentence:
        A person is training his horse for a competition.
        Response:
        n
        Paragraph:
        Children smiling and waving at camera
        Sentence:
        The kids are frowning
        Response:
        c
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
ENGLISH_THREE_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        A person on a horse jumps over a broken down airplane.
        Sentence:
        A person is training his horse for a competition.
        Response:
        n
        Paragraph:
        Children smiling and waving at camera
        Sentence:
        The kids are frowning
        Response:
        c
        Paragraph:
        A boy is jumping on skateboard in the middle of a red bridge.
        Sentence:
        The boy does a skateboarding trick.
        Response:
        e
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """


####################LCHAIM####################

####################LONG####################

ZERO_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות לתשובות:
        תשובה:
        מ
        תשובה:
        ס
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

LCH_ONE_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        במהלך המאה ה-20 חלו שינויים משמעותיים בתנאי העבודה במדינות המערב. אף על פי שלא חסרו בעיות נלוות, ניתן לראות את השינויים הללו באופן כללי כחיוביים: עבודת ילדים כמעט פסקה, השכר עלה, מספר שעות העבודה בשבוע פחת, מדיניות הפנסיה הפכה לסטנדרטית, הטבות נלוות התרבו, ודאגה לבריאות ולבטיחות בעבודה הפכה למחייבת. איסוף נתונים על תנאי העבודה הפך למדע מדויק הרבה יותר. במיוחד חלו התפתחויות חשובות בשיטות איסוף הנתונים. בנוסף, חלה התרחבות משמעותית במאמץ איסוף הנתונים, יותר אנשים היו מעורבים בלמידה על מקום העבודה; ולראשונה, החלו להתפרסם תוצאות. כתוצאה מכך, בסוף המאה, לא רק שרוב העובדים היו במצב טוב יותר מקודמיהם בתחילת המאה ה-20, אלא הם גם היו בעמדה להבין כיצד ומדוע זה היה המקרה. על ידי ניתוח קפדני של הנתונים הסטטיסטיים שהיו זמינים, שינויים ספציפיים במקום העבודה לא פחות מאשר בנוגע למושג מה העבודה צריכה לכלול הפכו ברורים. השינויים הבולטים ביותר בסביבת העבודה נגעו לגודל ולמבנה של כוח העבודה. בארצות הברית, למשל, גדל כוח העבודה מ-24 מיליון (כולל עובדים מגיל עשר ומעלה) ל-139 מיליון (מגיל 16 ומעלה), כמעט פי שישה, בהתאם לגידול באוכלוסייה הכללית. באותה עת, הרכב כוח העבודה השתנה מתעשיות שעיקרן ייצור חקלאי, כמו חקלאים ויערנים, לתעשיות שעיקרן מקצועות חופשיים, טכניים ובמיוחד שירותים. בתחילת המאה ה־20, 38% מכלל העובדים האמריקנים הועסקו בחקלאות, בסופה של אותה מאה, שיעור זה צנח לפחות מ־3%. באירופה, תהליך דומה התרחש. בשנות ה־30 של המאה ה־20, בכל מדינה אירופית, פרט לבריטניה ובלגיה, יותר מ־20% מהאוכלוסייה עסקו בחקלאות. בשנות ה־80, לעומת זאת, אוכלוסיית החקלאים בכל המדינות המפותחות, למעט מזרח אירופה, צנחה ל־10% ולעתים אף פחות מכך. באותה עת, החקלאות האינטנסיבית, שהשתמשה בטכניקות ממוכנות, צמצמה באופן דרמטי את מספר העובדים הנדרשים לעבודה. וכאן טמונה הבעיה. בעוד מקום העבודה הפך לסביבה בטוחה ופרודוקטיבית יותר, הרחק מתנאי העבודה הקשים של אבותינו, המעבר מחקלאות לעבודה מודרנית יצר גם אבטלה המונית במדינות רבות. בלב הבעיה עמד המעבר מהכפר לעיר. לאחר שאיבדו את פרנסתם, התקבצו אוכלוסיות האיכרים במספרים הולכים וגדלים בקהילות צפופות, שבהן שיעורי התעסוקה לא הצליחו להדביק את קצב ההגירה הפנימית. כתוצאה מכך, אלפים נותרו יושבים במעברות בפאתי הערים, ממתינים למשרות שאולי לעולם לא יגיעו. בעוד שתופעה זו (ועדיין) אופיינית למדינות העולם השלישי, ניתן היה להבחין בה גם בכמה ערים אמריקאיות, צרפתיות, אנגליות וגרמניות בסוף המאה ה־20. מנקודת מבט שונה וחיובית, במאה ה-20 נשים הפכו לחברות פעילות ונראות בכל תחומי שוק העבודה המערבי. בשנת 1900, רק 19% מהנשים האירופאיות בגיל העבודה השתתפו בכוח העבודה; בשנת 1999, נתון זה עלה ל-60%. בשנת 1900, רק 1% מעורכי הדין במדינה ו-6% מהרופאים היו נשים; לעומת זאת, הנתונים היו 29% ו-24% בשנת 1999. סקר שנערך לאחרונה בקרב בני נוער צרפתים, גברים ונשים כאחד, העלה כי למעלה מ-50% מהנשאלים סבורים כי בכל עבודה (למעט זו הכרוכה בשירות צבאי) נשים הן עובדות טובות יותר, משום שהן נוטות פחות להתרגז תחת לחץ, ופחות תחרותיות מגברים. השינוי האחרון והאולי משמעותי ביותר במקומות העבודה של המאה ה-20 היה הכנסת הטכנולוגיה. רשימת השיפורים הטכנולוגיים במקומות העבודה היא אינסופית: מכשירי תקשורת ומדידה, מחשבים בכל הצורות והגדלים, רנטגן, לייזרים, אורות ניאון, פלדת אל-חלד וכן הלאה וכן הלאה. שיפורים אלה הובילו לסביבת עבודה יצרנית ובטוחה יותר. יתרה מכך, העובדה שהרפואה השתפרה באופן דרמטי כל כך הובילה לעלייה בתוחלת החיים בקרב אוכלוסיות מערביות. בתורן, עובדים בגילאים שונים מאוד יכלו לעבוד כתף אל כתף, ולהמשיך בעבודתם במשך שנים רבות יותר. בסוף המאה ה-20, סביבת העבודה המערבית עברה שינויים ניכרים. באופן כללי, הן גברים והן נשים עבדו פחות שעות ביום במשך שנים רבות יותר בתנאים טובים יותר. עם זאת, כוחה של החקלאות נחלש כאשר חקלאים ויערנים עברו לערים כדי להרוויח משכורות גבוהות יותר כסטטיסטיקאים ורואי חשבון. עבור אלה שלא יכלו לעשות את המעבר הזה, החיים עם שחר המאה החדשה נראו פחות מושכים.',
        משפט:
        שיפורים ברפואה הובילו לכך שעובדים מרוויחים יותר לאורך זמן.
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

LCH_TWO_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        נער בן 13, גארת' ג'ונס, נלקח לתחנת המשטרה בדאונסטון ביום שבת 11 ביוני בחשד לגניבה מחנות מקומית. גארת' ג'ונס מכחיש את כל ההאשמות נגדו. ידוע גם ש: גארת' הוא יתום. קצין הביטחון של החנות נוטר טינה לגארת' משום שהוא החבר הטוב ביותר של בנו. גארת' אינו מופיע בסרטוני האבטחה של החנות. לפני שנתיים נתפס גארת' גונב מחסומי תנועה משטרתיים. החנות היתה עמוסה מאוד ביום שבת 11 ביוני. לגארת לא ניתנה קבלה על הסחורה שקנה. גארת נעצר לאחר שעזב את החנות.
        משפט:
        גארת קנה כמה סחורות בחנות.
        תשובה:
        מ
        פסקה:
        במהלך המאה ה-20 חלו שינויים משמעותיים בתנאי העבודה במדינות המערב. אף על פי שלא חסרו בעיות נלוות, ניתן לראות את השינויים הללו באופן כללי כחיוביים: עבודת ילדים כמעט פסקה, השכר עלה, מספר שעות העבודה בשבוע פחת, מדיניות הפנסיה הפכה לסטנדרטית, הטבות נלוות התרבו, ודאגה לבריאות ולבטיחות בעבודה הפכה למחייבת. איסוף נתונים על תנאי העבודה הפך למדע מדויק הרבה יותר. במיוחד חלו התפתחויות חשובות בשיטות איסוף הנתונים. בנוסף, חלה התרחבות משמעותית במאמץ איסוף הנתונים, יותר אנשים היו מעורבים בלמידה על מקום העבודה; ולראשונה, החלו להתפרסם תוצאות. כתוצאה מכך, בסוף המאה, לא רק שרוב העובדים היו במצב טוב יותר מקודמיהם בתחילת המאה ה-20, אלא הם גם היו בעמדה להבין כיצד ומדוע זה היה המקרה. על ידי ניתוח קפדני של הנתונים הסטטיסטיים שהיו זמינים, שינויים ספציפיים במקום העבודה לא פחות מאשר בנוגע למושג מה העבודה צריכה לכלול הפכו ברורים. השינויים הבולטים ביותר בסביבת העבודה נגעו לגודל ולמבנה של כוח העבודה. בארצות הברית, למשל, גדל כוח העבודה מ-24 מיליון (כולל עובדים מגיל עשר ומעלה) ל-139 מיליון (מגיל 16 ומעלה), כמעט פי שישה, בהתאם לגידול באוכלוסייה הכללית. באותה עת, הרכב כוח העבודה השתנה מתעשיות שעיקרן ייצור חקלאי, כמו חקלאים ויערנים, לתעשיות שעיקרן מקצועות חופשיים, טכניים ובמיוחד שירותים. בתחילת המאה ה־20, 38% מכלל העובדים האמריקנים הועסקו בחקלאות, בסופה של אותה מאה, שיעור זה צנח לפחות מ־3%. באירופה, תהליך דומה התרחש. בשנות ה־30 של המאה ה־20, בכל מדינה אירופית, פרט לבריטניה ובלגיה, יותר מ־20% מהאוכלוסייה עסקו בחקלאות. בשנות ה־80, לעומת זאת, אוכלוסיית החקלאים בכל המדינות המפותחות, למעט מזרח אירופה, צנחה ל־10% ולעתים אף פחות מכך. באותה עת, החקלאות האינטנסיבית, שהשתמשה בטכניקות ממוכנות, צמצמה באופן דרמטי את מספר העובדים הנדרשים לעבודה. וכאן טמונה הבעיה. בעוד מקום העבודה הפך לסביבה בטוחה ופרודוקטיבית יותר, הרחק מתנאי העבודה הקשים של אבותינו, המעבר מחקלאות לעבודה מודרנית יצר גם אבטלה המונית במדינות רבות. בלב הבעיה עמד המעבר מהכפר לעיר. לאחר שאיבדו את פרנסתם, התקבצו אוכלוסיות האיכרים במספרים הולכים וגדלים בקהילות צפופות, שבהן שיעורי התעסוקה לא הצליחו להדביק את קצב ההגירה הפנימית. כתוצאה מכך, אלפים נותרו יושבים במעברות בפאתי הערים, ממתינים למשרות שאולי לעולם לא יגיעו. בעוד שתופעה זו (ועדיין) אופיינית למדינות העולם השלישי, ניתן היה להבחין בה גם בכמה ערים אמריקאיות, צרפתיות, אנגליות וגרמניות בסוף המאה ה־20. מנקודת מבט שונה וחיובית, במאה ה-20 נשים הפכו לחברות פעילות ונראות בכל תחומי שוק העבודה המערבי. בשנת 1900, רק 19% מהנשים האירופאיות בגיל העבודה השתתפו בכוח העבודה; בשנת 1999, נתון זה עלה ל-60%. בשנת 1900, רק 1% מעורכי הדין במדינה ו-6% מהרופאים היו נשים; לעומת זאת, הנתונים היו 29% ו-24% בשנת 1999. סקר שנערך לאחרונה בקרב בני נוער צרפתים, גברים ונשים כאחד, העלה כי למעלה מ-50% מהנשאלים סבורים כי בכל עבודה (למעט זו הכרוכה בשירות צבאי) נשים הן עובדות טובות יותר, משום שהן נוטות פחות להתרגז תחת לחץ, ופחות תחרותיות מגברים. השינוי האחרון והאולי משמעותי ביותר במקומות העבודה של המאה ה-20 היה הכנסת הטכנולוגיה. רשימת השיפורים הטכנולוגיים במקומות העבודה היא אינסופית: מכשירי תקשורת ומדידה, מחשבים בכל הצורות והגדלים, רנטגן, לייזרים, אורות ניאון, פלדת אל-חלד וכן הלאה וכן הלאה. שיפורים אלה הובילו לסביבת עבודה יצרנית ובטוחה יותר. יתרה מכך, העובדה שהרפואה השתפרה באופן דרמטי כל כך הובילה לעלייה בתוחלת החיים בקרב אוכלוסיות מערביות. בתורן, עובדים בגילאים שונים מאוד יכלו לעבוד כתף אל כתף, ולהמשיך בעבודתם במשך שנים רבות יותר. בסוף המאה ה-20, סביבת העבודה המערבית עברה שינויים ניכרים. באופן כללי, הן גברים והן נשים עבדו פחות שעות ביום במשך שנים רבות יותר בתנאים טובים יותר. עם זאת, כוחה של החקלאות נחלש כאשר חקלאים ויערנים עברו לערים כדי להרוויח משכורות גבוהות יותר כסטטיסטיקאים ורואי חשבון. עבור אלה שלא יכלו לעשות את המעבר הזה, החיים עם שחר המאה החדשה נראו פחות מושכים.',
        משפט:
        שיפורים ברפואה הובילו לכך שעובדים מרוויחים יותר לאורך זמן.
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

LCH_THREE_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        נער בן 13, גארת' ג'ונס, נלקח לתחנת המשטרה בדאונסטון ביום שבת 11 ביוני בחשד לגניבה מחנות מקומית. גארת' ג'ונס מכחיש את כל ההאשמות נגדו. ידוע גם ש: גארת' הוא יתום. קצין הביטחון של החנות נוטר טינה לגארת' משום שהוא החבר הטוב ביותר של בנו. גארת' אינו מופיע בסרטוני האבטחה של החנות. לפני שנתיים נתפס גארת' גונב מחסומי תנועה משטרתיים. החנות היתה עמוסה מאוד ביום שבת 11 ביוני. לגארת לא ניתנה קבלה על הסחורה שקנה. גארת נעצר לאחר שעזב את החנות.
        משפט:
        גארת קנה כמה סחורות בחנות.
        תשובה:
        מ
        פסקה:
        במהלך המאה ה-20 חלו שינויים משמעותיים בתנאי העבודה במדינות המערב. אף על פי שלא חסרו בעיות נלוות, ניתן לראות את השינויים הללו באופן כללי כחיוביים: עבודת ילדים כמעט פסקה, השכר עלה, מספר שעות העבודה בשבוע פחת, מדיניות הפנסיה הפכה לסטנדרטית, הטבות נלוות התרבו, ודאגה לבריאות ולבטיחות בעבודה הפכה למחייבת. איסוף נתונים על תנאי העבודה הפך למדע מדויק הרבה יותר. במיוחד חלו התפתחויות חשובות בשיטות איסוף הנתונים. בנוסף, חלה התרחבות משמעותית במאמץ איסוף הנתונים, יותר אנשים היו מעורבים בלמידה על מקום העבודה; ולראשונה, החלו להתפרסם תוצאות. כתוצאה מכך, בסוף המאה, לא רק שרוב העובדים היו במצב טוב יותר מקודמיהם בתחילת המאה ה-20, אלא הם גם היו בעמדה להבין כיצד ומדוע זה היה המקרה. על ידי ניתוח קפדני של הנתונים הסטטיסטיים שהיו זמינים, שינויים ספציפיים במקום העבודה לא פחות מאשר בנוגע למושג מה העבודה צריכה לכלול הפכו ברורים. השינויים הבולטים ביותר בסביבת העבודה נגעו לגודל ולמבנה של כוח העבודה. בארצות הברית, למשל, גדל כוח העבודה מ-24 מיליון (כולל עובדים מגיל עשר ומעלה) ל-139 מיליון (מגיל 16 ומעלה), כמעט פי שישה, בהתאם לגידול באוכלוסייה הכללית. באותה עת, הרכב כוח העבודה השתנה מתעשיות שעיקרן ייצור חקלאי, כמו חקלאים ויערנים, לתעשיות שעיקרן מקצועות חופשיים, טכניים ובמיוחד שירותים. בתחילת המאה ה־20, 38% מכלל העובדים האמריקנים הועסקו בחקלאות, בסופה של אותה מאה, שיעור זה צנח לפחות מ־3%. באירופה, תהליך דומה התרחש. בשנות ה־30 של המאה ה־20, בכל מדינה אירופית, פרט לבריטניה ובלגיה, יותר מ־20% מהאוכלוסייה עסקו בחקלאות. בשנות ה־80, לעומת זאת, אוכלוסיית החקלאים בכל המדינות המפותחות, למעט מזרח אירופה, צנחה ל־10% ולעתים אף פחות מכך. באותה עת, החקלאות האינטנסיבית, שהשתמשה בטכניקות ממוכנות, צמצמה באופן דרמטי את מספר העובדים הנדרשים לעבודה. וכאן טמונה הבעיה. בעוד מקום העבודה הפך לסביבה בטוחה ופרודוקטיבית יותר, הרחק מתנאי העבודה הקשים של אבותינו, המעבר מחקלאות לעבודה מודרנית יצר גם אבטלה המונית במדינות רבות. בלב הבעיה עמד המעבר מהכפר לעיר. לאחר שאיבדו את פרנסתם, התקבצו אוכלוסיות האיכרים במספרים הולכים וגדלים בקהילות צפופות, שבהן שיעורי התעסוקה לא הצליחו להדביק את קצב ההגירה הפנימית. כתוצאה מכך, אלפים נותרו יושבים במעברות בפאתי הערים, ממתינים למשרות שאולי לעולם לא יגיעו. בעוד שתופעה זו (ועדיין) אופיינית למדינות העולם השלישי, ניתן היה להבחין בה גם בכמה ערים אמריקאיות, צרפתיות, אנגליות וגרמניות בסוף המאה ה־20. מנקודת מבט שונה וחיובית, במאה ה-20 נשים הפכו לחברות פעילות ונראות בכל תחומי שוק העבודה המערבי. בשנת 1900, רק 19% מהנשים האירופאיות בגיל העבודה השתתפו בכוח העבודה; בשנת 1999, נתון זה עלה ל-60%. בשנת 1900, רק 1% מעורכי הדין במדינה ו-6% מהרופאים היו נשים; לעומת זאת, הנתונים היו 29% ו-24% בשנת 1999. סקר שנערך לאחרונה בקרב בני נוער צרפתים, גברים ונשים כאחד, העלה כי למעלה מ-50% מהנשאלים סבורים כי בכל עבודה (למעט זו הכרוכה בשירות צבאי) נשים הן עובדות טובות יותר, משום שהן נוטות פחות להתרגז תחת לחץ, ופחות תחרותיות מגברים. השינוי האחרון והאולי משמעותי ביותר במקומות העבודה של המאה ה-20 היה הכנסת הטכנולוגיה. רשימת השיפורים הטכנולוגיים במקומות העבודה היא אינסופית: מכשירי תקשורת ומדידה, מחשבים בכל הצורות והגדלים, רנטגן, לייזרים, אורות ניאון, פלדת אל-חלד וכן הלאה וכן הלאה. שיפורים אלה הובילו לסביבת עבודה יצרנית ובטוחה יותר. יתרה מכך, העובדה שהרפואה השתפרה באופן דרמטי כל כך הובילה לעלייה בתוחלת החיים בקרב אוכלוסיות מערביות. בתורן, עובדים בגילאים שונים מאוד יכלו לעבוד כתף אל כתף, ולהמשיך בעבודתם במשך שנים רבות יותר. בסוף המאה ה-20, סביבת העבודה המערבית עברה שינויים ניכרים. באופן כללי, הן גברים והן נשים עבדו פחות שעות ביום במשך שנים רבות יותר בתנאים טובים יותר. עם זאת, כוחה של החקלאות נחלש כאשר חקלאים ויערנים עברו לערים כדי להרוויח משכורות גבוהות יותר כסטטיסטיקאים ורואי חשבון. עבור אלה שלא יכלו לעשות את המעבר הזה, החיים עם שחר המאה החדשה נראו פחות מושכים.',
        משפט:
        שיפורים ברפואה הובילו לכך שעובדים מרוויחים יותר לאורך זמן.
        תשובה:
        נ
        פסקה:
        במאבק בפשע. זוהי הראיה הפורנזית הנפוצה ביותר, לעתים קרובות עולה על שיטות זיהוי אחרות. בימים אלה, שיטות ישנות יותר של טביעות אצבעות בדיו, שיכולות לקחת שבועות, פינו את מקומן לטכניקות חדשות ומהירות יותר כמו סריקת לייזר של טביעות אצבעות, אך העקרונות נשארים זהים. לא משנה באיזו דרך תאספו ראיות טביעות אצבע, כל טביעת אצבע של כל אדם היא ייחודית. אז מה הופך את טביעות האצבע שלנו לשונות משל השכנים שלנו? מקום טוב להתחיל בו הוא להבין מהן טביעות אצבע וכיצד הן נוצרות. טביעת אצבע היא סידור של חריצי עור ושקעים בקצות האצבעות. עור מחוספס זה מתפתח במלואו במהלך התפתחות העובר, כאשר תאי העור גדלים ברחם האם. החריצים מסודרים לתוך דפוסים ונשארים זהים לאורך כל חייו של אדם. מאפיינים אנושיים נראים אחרים, כמו משקל וגובה, משתנים עם הזמן, בעוד טביעות אצבעות אינן. הסיבה לכך שכל טביעת אצבע היא ייחודית היא שכשגנים של תינוק משתלבים עם השפעות סביבתיות, כמו טמפרטורה, זה משפיע על הדרך שבה החריצים בעור גדלים. זה גורם לחריצים להתפתח בקצב שונה, להתכופף ולהתעקם לתוך דפוסים. כתוצאה מכך, לאף שני אנשים אין את אותן טביעות אצבע. אפילו תאומים זהים מחזיקים בטביעות אצבע שונות. לא קל לשרטט את מסלול התגלית של טביעת האצבע הייחודית. הרגע ההיסטורי שבו זה קרה אינו ידוע במדויק. עם זאת, ניתן לעקוב אחר השימוש בטביעות אצבעות אחורה למספר תרבויות עתיקות, כגון בבל וסין, שבהן הודפסו טביעות אצבע על לוחות חימר כדי לאשר עסקאות. האם אנשים באותה תקופה הבינו את מלוא הפוטנציאל של טביעות אצבעות לזיהוי הוא עניין אחר לגמרי. אין לדעת אם המעשה נתפס כדרך לאשר זהות או כמחווה סמלית לקשירת חוזה, שבו מתן טביעת אצבע היא כמו מתן מילה. למרות אי־הוודאות, היו מי שתרמו תרומה משמעותית לניתוח טביעות האצבע. ההיסטוריה מספרת לנו שרופא פרסי בן המאה ה־14 הצהיר הצהרה מוקדמת שלפיה אין שתי טביעות אצבע זהות. מאוחר יותר, במאה ה־17, חקר הרופא האיטלקי מרקלו מלפיגי את הצורות המבחינות של לולאות וספירלות בטביעות אצבע. לכבודו, קראה לו מאוחר יותר הקהילה הרפואית שכבה של עור על שמו. אולם היה זה עובד של חברת הודו המזרחית, ויליאם הרשל, שהבין את הפוטנציאל האמיתי של טביעות האצבע. הוא לקח טביעות אצבעות מאנשי המקום כחתימה על חוזים, כדי למנוע הונאה. סקרנותו לגבי טביעות האצבעות דחפה אותו לחקור אותן במשך עשרים השנים הבאות. הוא פיתח את התיאוריה שטביעות האצבעות הן ייחודיות לאדם ואינן משתנות כלל לאורך החיים. בשנת 1880 הציע הנרי פולדס כי טביעות אצבעות יכולות לשמש לזיהוי פושעים מורשעים. הוא כתב לצ"ר דרווין לקבלת עצה, והרעיון הועבר לקרובו של דרווין, סר פרנסיס גלטון. גלטון פרסם בסופו של דבר מחקר מעמיק של מדע טביעות האצבע בשנת 1892. אף על פי שהעובדה שכל אדם יש דפוס טביעת אצבע ייחודי לחלוטין תועדה ואושרה במשך זמן רב, ידע זה לא נוצל לזיהוי פלילי עד תחילת המאה ה-20. בעבר שימשו סימני צריבה, קעקוע ועיוות לסימון הפושע על מה שהוא. במדינות מסוימות היו כורתים את ידי הגנבים. צרפת צרבה פושעים בסמל הפרחים. הרומאים קעקעו חיילים שכירים כדי למנוע מהם להפוך לעריקים. במשך שנים רבות סרבו סוכנויות משטרה במערב להשתמש בטביעות אצבעות, והעדיפו את השיטה הפופולרית של אותה תקופה, שיטת ברטיון, שבה נרשמו מידות של חלקי גוף מסוימים כדי לזהות פושע. נקודת המפנה הייתה בשנת 1903, כאשר אסיר בשם וויל ווסט הושם במתקן הכליאה הפדרלי של לבנורת. באופן מפתיע, לוויל היו כמעט אותן מידות ברטילון כמו לאסיר אחר ששהה באותו מתקן, ששמו היה ויליאם ווסט. הדבר היחיד שהבדיל ביניהם היה טביעות האצבעות שלהם. מנקודה זו ואילך, טביעות אצבעות הפכו לסטנדרט לזיהוי פלילי. טביעות אצבעות היו שימושיות בזיהוי אנשים בעלי היסטוריה של פשע, שהיו רשומים במאגר מידע. עם זאת, במצבים שבהם העבריין לא היה במאגר, ולא היו עדים לפשע, המערכת לא הצליחה. כימיית טביעות אצבע היא טכנולוגיה חדשה שיכולה לעבוד לצד טביעות אצבע מסורתיות כדי למצוא רמזים רבים יותר מאי פעם. מטביעות אצבע אורגניות שנשארו מאחור, מדען יכול לומר אם האדם הוא ילד, מבוגר, אדם בוגר או מעשן, ועוד הרבה יותר. מסתבר, אחרי כל השנים האלה, האצבעות ממשיכות להצביע על הדרך.
        משפט:
        חיילים רומאים קועקעו כדי למנוע מהם לבצע פשעים אלימים.
        תשובה:
        ס
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

####################SHORT####################


LCH_ONE_SHORT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        ממשלת דלהי החליטה לפרוס מרשלים במאה אוטובוסים של תאגיד התחבורה של דלהי (DTC) ונקודות חשוכות כדי להפוך את התחבורה הציבורית והמקומות הציבוריים לבטוחים יותר עבור נשות הבירה.
        משפט:
        נשים ירגישו בטוחות יותר בתחבורה ציבורית ובמקומות ציבוריים.
        תשובה:
        מ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

LCH_TWO_SHORT_PROMPT = """ (ס,מ,נ)אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        ממשלת דלהי החליטה לפרוס מרשלים במאה אוטובוסים של תאגיד התחבורה של דלהי (DTC) ונקודות חשוכות כדי להפוך את התחבורה הציבורית והמקומות הציבוריים לבטוחים יותר עבור נשות הבירה.
        משפט:
        נשים ירגישו בטוחות יותר בתחבורה ציבורית ובמקומות ציבוריים.
        תשובה:
        מ
        פסקה:
        ארין בת שתים עשרה. במשך שלוש שנים היא מבקשת מהוריה כלב. הוריה אמרו לה שהם מאמינים שכלב לא יהיה מאושר בדירה, אבל הם נתנו לה רשות ללדת ציפור. ארין עדיין לא החליטה איזה סוג של ציפור היא רוצה לקבל.,
        משפט:
        ארין והוריה היו רוצים לעבור דירה.
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

LCH_THREE_SHORT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        ממשלת דלהי החליטה לפרוס מרשלים במאה אוטובוסים של תאגיד התחבורה של דלהי (DTC) ונקודות חשוכות כדי להפוך את התחבורה הציבורית והמקומות הציבוריים לבטוחים יותר עבור נשות הבירה.
        משפט:
        נשים ירגישו בטוחות יותר בתחבורה ציבורית ובמקומות ציבוריים.
        תשובה:
        מ
        פסקה:
        ארין בת שתים עשרה. במשך שלוש שנים היא מבקשת מהוריה כלב. הוריה אמרו לה שהם מאמינים שכלב לא יהיה מאושר בדירה, אבל הם נתנו לה רשות ללדת ציפור. ארין עדיין לא החליטה איזה סוג של ציפור היא רוצה לקבל.
        משפט:
        ארין והוריה היו רוצים לעבור דירה.
        תשובה:
        נ
        פסקה:
        לאחר שסיפקה גפ"מ בצילינדרים קלים לנשיאה של 5 ק"ג, הממשלה השיקה בקבוקים של 2 ק"ג בחנויות קירנה המקומיות והציגה הזמנה מקוונת של חיבורים חדשים לדלק בישול מסובסד.
        משפט:
        התוכנית תועיל במיוחד לאנשים הכפריים ולעניים שאינם יכולים להרשות לעצמם לשלם מחיר של גליל 14.2 ק"ג או אפילו 5 ק"ג
        תשובה:
        ס
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""


#################SNLI################

SNLI_ONE_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        פשוט תמשיך במה שאתה עושה עכשיו.
        משפט:
        המשך במטלה הנוכחית שיש לך.
        תשובה:
        מ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

SNLI_TWO_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        פשוט תמשיך במה שאתה עושה עכשיו.
        משפט:
        המשך במטלה הנוכחית שיש לך.
        תשובה:
        מ
        פסקה:
        אתה לא חושב שזה יכול להיות?
        משפט:
        אתה לא חושב שזה יכול להיות האיש שאנחנו מחפשים?
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""

SNLI_THREE_SHOT_PROMPT = """ אני אתן לך פסקה ומשפט ואתה תצטרך לסווג האם המשפט סותר את הנאמר הפסקה (ס) , נגזר מהנאמר מהפסקה (מ) או לא סותר את הנאמר בפסקה ולא נגזר מהאמר בפסקה (נ) עליך לענות באמצעות אחת האותיות המייצגות את הקשר של המשפט לטקסט בלבד!
        דוגמאות:
        פסקה:
        פשוט תמשיך במה שאתה עושה עכשיו.
        משפט:
        המשך במטלה הנוכחית שיש לך.
        תשובה:
        מ
        פסקה:
        אתה לא חושב שזה יכול להיות?
        משפט:
        אתה לא חושב שזה יכול להיות האיש שאנחנו מחפשים?
        תשובה:
        נ
        פסקה:
        בחוסר רצון מסוים, גרוז הרשה לי ללכת לרחרח.
        משפט:
        גרוז אמר לי שאני לא יכול לבדוק.
        תשובה:
        נ
        ענה על הפסקה והמשפט הבאים
        פסקה:
        {p}
        משפט:
        {h}
        תשובה:"""


################TRANSLATED_BACK_LCHAIM #########################


BACK_TO_ENGLISH_ONE_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        The Delhi government decided to deploy marshals in 100 Delhi Transport Corporation (DTC) buses and dark spots to make public transport and public places safer for the women of the capital.
        Sentence:
        Delhi residents will buy two cars.
        Response:
        n
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
BACK_TO_ENGLISH_TWO_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        The Delhi government decided to deploy marshals in 100 Delhi Transport Corporation (DTC) buses and dark spots to make public transport and public places safer for the women of the capital.
        Sentence:
        Delhi residents will buy two cars.
        Response:
        n
        Paragraph:
        Erin is twelve. For three years she has been asking her parents for a dog. Her parents told her they believed a dog would not be happy in the apartment, but they gave her permission to have a bird. Erin has not yet decided what kind of bird she wants to get.
        Sentence:
        Erin and her parents live in the apartment.
        Response:
        e
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
BACK_TO_ENGLISH_THREE_SHOT_PROMPT_SHORT = """I will give you a paragraph and a sentence, and you will need to classify whether the sentence contradicts the paragraph (C), is entailed by the paragraph (E), or neither contradicts nor is entailed by the paragraph - means natural (N). You should respond using only one of the letters representing the relationship of the sentence to the text.
        examples:
        Paragraph:
        The Delhi government decided to deploy marshals in 100 Delhi Transport Corporation (DTC) buses and dark spots to make public transport and public places safer for the women of the capital.
        Sentence:
        Delhi residents will buy two cars.
        Response:
        n
        Paragraph:
        Erin is twelve. For three years she has been asking her parents for a dog. Her parents told her they believed a dog would not be happy in the apartment, but they gave her permission to have a bird. Erin has not yet decided what kind of bird she wants to get.
        Sentence:
        Erin and her parents live in the apartment.
        Response:
        e
        Paragraph:
        After supplying LPG in lightweight 5-kg cylinders, the government launched 2kg bottles in local Kirana stores and introduced online ordering of new connections to subsidised cooking fuel.
        Sentence:
        The scheme will particularly benefit the rural and poor people who cannot afford to pay a price of a 14.2 kg roll or even 5 kg.
        Response:
        c
        Answer for the following paragraph and sentence
        Paragraph:
        {p}
        Sentence:
        {h}
        Response:
        """
CONTROL_PRONPTS = [ENGLISH_ZERO_SHOT_PROMPT, ENGLISH_ONE_SHOT_PROMPT, ENGLISH_TWO_SHOT_PROMPT, ENGLISH_THREE_SHOT_PROMPT] 
CONTROL_PRONPTS_SHORT = [ENGLISH_ZERO_SHOT_PROMPT, ENGLISH_ONE_SHOT_PROMPT_SHORT, ENGLISH_TWO_SHOT_PROMPT_SHORT, ENGLISH_THREE_SHOT_PROMPT_SHORT] 

SNLI_PROMPTS = [ZERO_SHOT_PROMPT, SNLI_ONE_SHOT_PROMPT, SNLI_TWO_SHOT_PROMPT, SNLI_THREE_SHOT_PROMPT]

LCHAIM_PRONPTS = [ZERO_SHOT_PROMPT, LCH_ONE_SHOT_PROMPT, LCH_TWO_SHOT_PROMPT, LCH_THREE_SHOT_PROMPT]
LCHAIM_PRONPTS_SHORT_EDITION_WITH_SHORT_LCHAIM = [ZERO_SHOT_PROMPT, LCH_ONE_SHORT_PROMPT, LCH_TWO_SHORT_PROMPT, LCH_THREE_SHORT_PROMPT]
LCHAIM_PRONPTS_SHORT_EDITION = SNLI_PROMPTS ## !
TRANSLATED_BACK_LCHAIM = [ENGLISH_ZERO_SHOT_PROMPT, BACK_TO_ENGLISH_ONE_SHOT_PROMPT_SHORT, BACK_TO_ENGLISH_TWO_SHOT_PROMPT_SHORT, BACK_TO_ENGLISH_THREE_SHOT_PROMPT_SHORT] 

# Test

## Functions

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
from datetime import datetime
import seaborn as sns
import torch
import json
import os


def get_prompt(n_shots, is_SNLI=False, is_CONTROL=False, is_CONTROL_SHORT=False, is_LCHAIM=False, is_LCHAIM_SHORT=False, is_LCHAIM_SHORT_2=False, is_back_to_english=False):
    if n_shots > 3:
        raise Exception('To much shots')
    if is_SNLI:
        return SNLI_PROMPTS[n_shots]
    if is_back_to_english:
        return TRANSLATED_BACK_LCHAIM[n_shots]
    if is_LCHAIM:
        return LCHAIM_PRONPTS[n_shots]
    if is_LCHAIM_SHORT:
        return LCHAIM_PRONPTS_SHORT_EDITION[n_shots]
    if is_LCHAIM_SHORT_2:
        return LCHAIM_PRONPTS_SHORT_EDITION_WITH_SHORT_LCHAIM[n_shots]
    if is_CONTROL:
        return CONTROL_PRONPTS[n_shots]
    if is_CONTROL_SHORT:
        return CONTROL_PRONPTS_SHORT[n_shots]
    return None
    

def infer_nli(premise, hypothesis, prompt):
    messages = [
    {"role": "user", "content":prompt.format(p=premise, h=hypothesis)}
    ]
    encoded = None
    if TESTED_MODEL == 'gemma':
        encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, max_length=8192).to(device)
    else:
        encoded = tokenizer.apply_chat_template(messages, return_tensors="pt").to(device)
    generated_ids = model.generate(encoded, max_new_tokens=1, do_sample=False, temperature=0, pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.batch_decode(generated_ids)
    return decoded[0]


def calculate_scores(results, average='macro', heb=True):
    true_labels = [r['label'] for r in results]
    predicted_labels = None
    if TESTED_MODEL == 'gemma' :
        predicted_labels = [r['prediction'].split('model\n')[1].strip() for r in results] 
    else:
        predicted_labels = [r['prediction'].split('[/INST]')[1].strip() for r in results]
    d = {}
    if heb:
        d = {'ס':'c', 'מ':'e', 'נ':'n'}
    converted_predictions = [d.get(pred, pred).lower() for pred in predicted_labels]
    
    accuracy = accuracy_score(true_labels, converted_predictions)    
    precision = precision_score(true_labels, converted_predictions, average=average)
    recall = recall_score(true_labels, converted_predictions, average=average)
    
    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall}


def calculate_scores_confiusion_matrix(results, model_name, file, n_of_shots, average='macro' , heb=True):
    true_labels = [r['label'] for r in results]
    predicted_labels = None
    if 'gemma' in model_name:
        predicted_labels = [r['prediction'].split('model\n')[1].strip() for r in results] 
    else:
        predicted_labels = [r['prediction'].split('[/INST]')[1].strip() for r in results]
    d = {}
    if heb:
        d = {'ס':'c', 'מ':'e', 'נ':'n'}
    
    converted_predictions = [d.get(pred, pred).lower() for pred in predicted_labels]
    # print("XXXX", sum(label not in valid_labels for label in converted_predictions)) ## sanity check
    
    labels = [label for _, label in sorted(d.items(), key=lambda item: item[1])] 
    accuracy = accuracy_score(true_labels, converted_predictions)    
    cm = confusion_matrix(true_labels, converted_predictions, labels=list(d.values()))
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    report = classification_report(true_labels, converted_predictions, target_names=labels, digits=4)
    plt.gcf().text(0.25, -0.3, "Classification Report:\n" + report, fontsize=12, ha="left")
    # file_path = f"Lchaim_project/AWS_RESULTS_IN_PICTURES/{model_name}_{file}_{n_of_shots}_shots_english_res.png"
    # plt.savefig(file_path, bbox_inches='tight')
    plt.show()
    return accuracy, classification_report(true_labels, converted_predictions, target_names=labels, digits=4, output_dict=True)


In [ ]:
prompt_flags = {
        "SNLI": {"is_SNLI": True},
        "CONTROL": {"is_CONTROL": True},
        "CONTROL_SHORT": {"is_CONTROL_SHORT": True},
        "LCHAIM": {"is_LCHAIM": True},
        "LCHAIM_SHORT": {"is_LCHAIM_SHORT": True},
        "LCHAIM_SHORT_2": {"is_LCHAIM_SHORT_2": True},
        "BACK_TO_ENGLISH": {"is_back_to_english": True}
    }

def read_json(path):
    data = []
    with open(path, 'r') as f:
        if path.endswith('jsonl'):
            number_of_not_working = 0
            for line in f:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    number_of_not_working += 1
                    continue
            if number_of_not_working > 0:
                print(f"Couldn't read {number_of_not_working} rows from '{path}'")
        elif path.endswith('json'):
            try:
                data = json.loads(f.read())
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format in file '{path}': {e}")
        else:
            raise ValueError(f"Invalid json_type: {path.split('.')[-1]}. Must be 'json' or 'jsonl'.")
    if not data:
        raise IOError(f"An error occurred while loading the file '{path}'")

    return data

def _write_results(results, path):
    with open(path, 'w') as f:
        f.write(json.dumps(results))


def test_model(files_to_test, shots_list, prompt_type, output_path=None, save_results=True):
    print(f"Testing {TESTED_MODEL} model...")
    results_dict = {}
    date = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

    if prompt_type not in prompt_flags:
        raise ValueError(f"Invalid prompt_type: {prompt_type}. Must be one of {list(prompt_flags.keys())}.")
    prompt_kwargs = prompt_flags[prompt_type]
    
    for file in files_to_test:
        results_dict[file] = {}
        for n_of_shots in shots_list:
            prompt = get_prompt(n_shots=n_of_shots, **prompt_kwargs)
            results_dict[file][f'{n_of_shots}_shots'] = {}
            results_dict[file][f'{n_of_shots}_shots']['results'] = []
            test_data = read_json(file)
            
            with torch.no_grad():
                for example in test_data:
                    premise = example['premise']
                    hypothesis = example['hypothesis']
                    label = example.get('label', None)[0]
                    predicted_class = infer_nli(premise, hypothesis, prompt)
                    results_dict[file][f'{n_of_shots}_shots']['results'].append({
                        'premise': premise,
                        'hypothesis': hypothesis,
                        'label': label,
                        'prediction': predicted_class,
                    })
            accuracy, report = calculate_scores_confiusion_matrix(results_dict[file][f'{n_of_shots}_shots']['results'] , 'gemma', file, n_of_shots)

    if save_results:
        if output_path:
            output_file = output_path    
        else:
            output_file = f'AWS_{TESTED_MODEL}_{prompt_type}_LCHAIM_PRONPTS_SHORT_EDITION_WITH_SHORT_LCHAIM_results_dicts_with_1_2_and_3_short_Shots_{formatted_date}.json'
        _write_results(results_dict, output_path)
        print(f'Inference completed. Results saved to {output_file}')
    else:
        print('Inference completed. Please note that the results did not saved')
    
    return results_dict  
            


# ConTRoL test

In [ ]:
files = ['control_dev.jsonl', 'control_test.jsonl', 'control_train.jsonl']
n_of_shots = [0,1,2,3]

In [ ]:
ctrl_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="CONTROL")

In [ ]:
ctrl_short_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="CONTROL_SHORT")

# LCHAIM test

In [ ]:
files = ['aws_heb_dev.json', 'aws_heb_test.json', 'aws_heb_train.json']
n_of_shots = [0,1,2,3]

In [ ]:
lchaim_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="LCHAIM")

In [ ]:
lchaim_short_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="LCHAIM_SHORT")

In [ ]:
lchaim_short_results_with_lchaim_prems = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="LCHAIM_SHORT_2")

## ENGLISH translation test

In [ ]:
files = ['eng_lchaim_test.json']
n_of_shots = [0,1,2,3]

In [ ]:
english_Lchiam_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="BACK_TO_ENGLISH")

# Heb SNLI Test

In [ ]:
files = ['HebNLI_test.json']
n_of_shots = [0,1,2,3]

In [ ]:
snli_results = test_model(files_to_test=files, shots_list=n_of_shots, prompt_type="SNLI")

## Get results as CSV

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json

def get_results_dict(results_files):
    results_dicts = []
    for test_model, file_path in results_files:
        with open(file_path, 'r') as f:
            results_dicts.append({'tested_model': test_model, 'res': json.loads(f.read())})
    return results_dicts

def combine_results_to_csv(results_files, tested_files):
    results_dicts = get_results_dict(results_files)
    models = [entry['tested_model'] for entry in results_dicts]

    data = []
    for i, model in enumerate(models):
        for file in tested_files:
            for n_of_shots in [0, 1, 2, 3]:
                accuracy, report = calculate_scores_confiusion_matrix(
                    results=results_dicts[i]['res'][file][f'{n_of_shots}_shots']['results'],
                    model_name=results_dicts[i]['tested_model'],
                    file=file,
                    n_of_shots=n_of_shots
                )
                data.append({
                    "Model": results_dicts[i]['tested_model'],
                    "File": file,
                    "Shot": f"{n_of_shots} shot",
                    "Accuracy": round(report['accuracy'], 4),
                    "c Precision": round(report['c']['precision'], 4),
                    "c Recall": round(report['c']['recall'], 4),
                    "c f1-score": round(report['c']['f1-score'], 4),
                    "c support": report['c']['support'],
                    "e Precision": round(report['e']['precision'], 4),
                    "e Recall": round(report['e']['recall'], 4),
                    "e f1-score": round(report['e']['f1-score'], 4),
                    "e support": report['e']['support'],
                    "n Precision": round(report['n']['precision'], 4),
                    "n Recall": round(report['n']['recall'], 4),
                    "n f1-score": round(report['n']['f1-score'], 4),
                    "n support": report['n']['support'],
            })
    
    df = pd.DataFrame(data)
    return df

